# Cobot task explosion and reconciliation

Explodes the `OrderJobTask` / `OrderJobResultTask` JSON lists in
`cobot_jsons.parquet` into one row per task, labels each field by measured
overlap with known ID domains, and reconciles robot quantities against
`picking`.

Both sides are cleartext since the re-pull, so the hash crosswalks are retired
(D86, D100). The only shape fix needed is the cobot's `S`-prefixed slot against
`picking.slotid`, built in §2.

Writes `cobot_order_tasks.parquet`, `cobot_result_tasks.parquet`,
`cobot_reconciliation_told.parquet`, `cobot_reconciliation_did.parquet`.

In [1]:
# Load the cobot JSON payloads and the clean reference tables (slots, SKU
# bridge, picking, workers), and print how many rows carry each of the two
# task lists.
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from wms_rpt import explode_tasks, identify_fields, reconcile_with_picking

CLEAN = Path("data/clean")
cj = pd.read_parquet(CLEAN / "cobot_jsons.parquet")
picking = pd.read_parquet(CLEAN / "picking.parquet")
refs = {t: pd.read_parquet(CLEAN / f"{t}.parquet")
        for t in ("slots", "shippingintent_sku", "picking", "workers")}
print(f"cobot rows: {len(cj):,}")
for col in ("json_jobtasks_orderjobtask", "json_jobtasks_orderjobresulttask"):
    print(f"{col}: {cj[col].notna().sum():,} populated")

cobot rows: 1,212,895
json_jobtasks_orderjobtask: 145,709 populated
json_jobtasks_orderjobresulttask: 614,299 populated


## 1. Explode both lists

One row per task. Nested sub-objects stay as JSON strings.

In [2]:
# Explode the two JSON lists into flat tables, one row per task.
# OrderJobTask is what the robot was TOLD to do; OrderJobResultTask is what it
# reported actually DOING. Nested sub-objects stay as JSON strings.
order_tasks  = explode_tasks(cj, "json_jobtasks_orderjobtask")
result_tasks = explode_tasks(cj, "json_jobtasks_orderjobresulttask")
for name, td in [("OrderJobTask", order_tasks), ("OrderJobResultTask", result_tasks)]:
    print(f"{name}: {len(td):,} tasks x {td.shape[1]} cols")
    print("   fields:", [c for c in td.columns
                         if c not in ("pk","pickticket_id","eventtype","ts","appname")])
display(order_tasks.head(3))
display(result_tasks.head(3))

OrderJobTask: 1,452,960 tasks x 28 cols
   fields: ['jobtaskid', 'eventaction', 'orderid', 'orderlineid', 'custowner', 'tasktype', 'tasksequence', 'tasksubsequence', 'tasktravelpriority', 'tasklocation', 'taskqty', 'itemno', 'itemupc', 'itemdesc', 'itemsize', 'itemimageurl', 'custom1', 'itemlength', 'itemwidth', 'itemheight', 'itemweight', 'counteligible', '_task_seq']
OrderJobResultTask: 1,186,131 tasks x 24 cols
   fields: ['jobtaskid', 'orderid', 'orderlineid', 'custowner', 'taskstatus', 'tasktype', 'tasklocation', 'taskqty', 'execqty', 'execuser', 'execdate', 'execrobot', 'itemno', 'custom1', 'lotno', 'serialno', 'exceptioncode', 'exceptionreason', '_task_seq']


,pk,pickticket_id,eventtype,ts,appname,jobtaskid,eventaction,orderid,orderlineid,custowner,...,itemdesc,itemsize,itemimageurl,custom1,itemlength,itemwidth,itemheight,itemweight,counteligible,_task_seq
0,27979821,341CCCD0-DF44-4C3E-8443-833F9BC244C6,UPDATE,2025-12-01 14:01:57,LOCUSCONSUMER,347801_S144367_5,NEW,341cccd0-df44-4c3e-8443-833f9bc244c6,1,FRAGRANCENET,...,PACO RABANNE 1 MILLION; EDT SPRAY 0.27 OZ (TRA...,0.27 OZ,http://www.fragrancenet.com/images/photos/250x...,14/L/4/S144367,NaN,NaN,NaN,NaN,<NA>,0
1,27979823,341CCCD0-DF44-4C3E-8443-833F9BC244C6,UPDATE,2025-12-01 14:01:57,LOCUSWORKER,347801_S144367_5,NEW,341cccd0-df44-4c3e-8443-833f9bc244c6,1,FRAGRANCENET,...,PACO RABANNE 1 MILLION; EDT SPRAY 0.27 OZ (TRA...,0.27 OZ,http://www.fragrancenet.com/images/photos/250x...,14/L/4/S144367,NaN,NaN,NaN,NaN,<NA>,0
2,27979858,0D7670AE-3034-4018-B029-EC8B878BA107,UPDATE,2025-12-01 14:02:21,LOCUSCONSUMER,118286_S341205_19,NEW,0d7670ae-3034-4018-b029-ec8b878ba107,16,FRAGRANCENET,...,MICHAEL JORDAN; COLOGNE SPRAY 3.4 OZ,3.4 OZ,http://www.fragrancenet.com/images/photos/250x...,34/E/1/S341205,NaN,NaN,NaN,NaN,<NA>,0


,pk,pickticket_id,eventtype,ts,appname,jobtaskid,orderid,orderlineid,custowner,taskstatus,...,execuser,execdate,execrobot,itemno,custom1,lotno,serialno,exceptioncode,exceptionreason,_task_seq
0,27979732,8480B7DD-B6BD-49BD-B399-7DB6E2CA073E,PICK,2025-12-01 14:01:10,LOCUSCONSUMER,8480b7dd-b6bd-49bd-b399-7db6e2ca073e_217589_9,8480b7dd-b6bd-49bd-b399-7db6e2ca073e,9.0,FRAGRANCENET,COMPLETED,...,5341,2025-12-01T14:01:07.283-05:00,P3-6155,217589.0,52/L/3/S523365,[{}],[{}],NaN,NaN,0
1,27979733,0BCA3127-2921-4C0C-916B-79A5CE231461,PICK,2025-12-01 14:01:10,LOCUSCONSUMER,0bca3127-2921-4c0c-916b-79a5ce231461_484053_19,0bca3127-2921-4c0c-916b-79a5ce231461,19.0,FRAGRANCENET,COMPLETED,...,93289,2025-12-01T14:01:08.483-05:00,P3-13215,484053.0,51/I/6/S516293,[{}],[{}],NaN,NaN,0
2,27979735,4EE3232C-CF8D-4E49-928C-EEB1E47305DE,PICK,2025-12-01 14:01:15,LOCUSCONSUMER,4ee3232c-cf8d-4e49-928c-eeb1e47305de_306105_4,4ee3232c-cf8d-4e49-928c-eeb1e47305de,4.0,FRAGRANCENET,COMPLETED,...,4674,2025-12-01T14:01:11.103-05:00,P3-12573,306105.0,419/D/1/S191173,[{}],[{}],NaN,NaN,0


## 2. Identify fields by evidence

A string field is labelled a reference when its values overlap a known ID
domain; small named integers are labeled quantities. `attribute` means no
special role was detected.

In [3]:
# Pull the slot out of custom1, then label every column of both task tables
# by checking its values against the reference tables — is it an item id, a
# slot, a badge, a quantity, or just an attribute?
items_x  = pd.read_parquet(CLEAN / "items.parquet")
slots_x  = pd.read_parquet(CLEAN / "slots.parquet")
workers_x = pd.read_parquet(CLEAN / "workers.parquet")

# The cobot writes the slot inside custom1 as S523365 (the path looks like
# 52/L/3/S523365); picking has the bare digits 523365. reconcile_with_picking
# strips the S itself, so slot_xw stays None. The assert makes sure most cobot
# slots really do match a picking slot — if they did not, the reconciliation
# would quietly fall back to ticket grain and the agreement figures would look
# better than they are.
for td in (order_tasks, result_tasks):
    td["_slot"] = td["custom1"].astype(str).str.extract(r"(S\d+)$")

slot_xw = None   # no crosswalk needed now; it was only for the old hashed extracts

_task_slots = pd.unique(pd.concat([order_tasks["_slot"], result_tasks["_slot"]])
                          .dropna().astype(str))
_bare = {t[1:] if t.startswith("S") else t for t in _task_slots}
_pick_slots = set(picking["slotid"].astype(str).str.strip())
_hit = len(_bare & _pick_slots)
print(f"cobot slots {len(_bare):,} | matching a picking slot: "
      f"{_hit:,} ({_hit/max(len(_bare),1):.1%})")
assert _hit / max(len(_bare), 1) > 0.9, \
    "slot overlap below 90% - check the cobot slot format before trusting this"

refs_plain = {"items":   items_x[["itemid"]],
              "slots":   slots_x[["slotid"]],
              "workers": workers_x[["badge"]],
              "picking": picking[["pickticket_id", "shippingintentline_id"]]}

ident_order  = identify_fields(order_tasks,  refs_plain)
ident_result = identify_fields(result_tasks, refs_plain)
print("OrderJobTask fields:")
display(ident_order)
print("OrderJobResultTask fields:")
display(ident_result)

cobot slots 28,217 | matching a picking slot: 27,851 (98.7%)
OrderJobTask fields:


,column,identified_as,evidence
0,pk,attribute,
1,pickticket_id,attribute,
2,eventtype,attribute,
3,ts,attribute,
4,appname,attribute,
5,jobtaskid,attribute,
6,eventaction,attribute,
7,orderid,attribute,
8,orderlineid,attribute,
9,custowner,attribute,


OrderJobResultTask fields:


,column,identified_as,evidence
0,pk,attribute,
1,pickticket_id,picking.pickticket_id_ref,100% of sampled values in picking.pickticket_id
2,eventtype,attribute,
3,ts,attribute,
4,appname,attribute,
5,jobtaskid,attribute,
6,orderid,picking.pickticket_id_ref,100% of sampled values in picking.pickticket_i...
7,orderlineid,attribute,
8,custowner,attribute,
9,taskstatus,attribute,


## 3. Told vs did — reconcile against `picking`

TOLD is `OrderJobTask.taskqty`; DID is `OrderJobResultTask.execqty`.
Re-emissions are deduped on `(pickticket_id, jobtaskid)` — `jobtaskid` is not
globally unique — and PACK tasks are dropped.

The order stream is a ledger, not a snapshot: negative `taskqty` rows are
adjustments against an earlier instruction.

The pre-D100 ticket-grain agreement figures are withdrawn and need
re-recording in the register from this run.

In [4]:
# Reconcile TOLD (OrderJobTask.taskqty) and DID (OrderJobResultTask.execqty)
# against picking, print the agreement figures, and save the four task and
# reconciliation tables. `show` is just a printer for a reconciliation result.
def show(rec, label):
    if rec is None:
        print(f"{label}: no usable quantity field — see the identification report.")
        return
    print(f"\n=== {label} ===")
    for k, v in rec.attrs.items():
        print(f"  {k}: {v:,}" if isinstance(v, int) else f"  {k}: {v}")
    print(f"  {rec.attrs.get('grain','ticket')} pairs reconciled : {len(rec):,}")
    print(f"  robot units == WMS ordered units : "
          f"{(rec.units_diff_vs_ordered == 0).mean():.1%}")
    print(f"  robot units == WMS picked units  : "
          f"{(rec.units_diff_vs_picked == 0).mean():.1%}")
    # tasks.py records the grain as "(ticket, slot)" or "ticket". If it fell back
    # to ticket grain the slots did not line up and I should not trust the numbers.
    if not str(rec.attrs.get("grain", "")).startswith("(ticket, slot)"):
        print("  ⚠︎ reconciled at TICKET grain — the slot columns didn't line up.")

# The order stream is a running ledger, not a snapshot: a negative taskqty is
# a correction to an earlier instruction. A plain sum would overstate TOLD, so
# I count the negatives and show which event action they arrive with.
if "taskqty" in order_tasks.columns:
    q = pd.to_numeric(order_tasks["taskqty"], errors="coerce")
    n_neg = int((q < 0).sum())
    print(f"negative-quantity instruction tasks: {n_neg:,} "
          f"({n_neg / max(len(order_tasks), 1):.2%})")
    if n_neg and "eventaction" in order_tasks.columns:
        print("  by eventaction:")
        print(order_tasks.loc[q < 0, "eventaction"]
              .value_counts().head(8).to_string())

rec_told = reconcile_with_picking(order_tasks, picking, ident_order,
                                  slot_xw=slot_xw)
show(rec_told, "TOLD — OrderJobTask.taskqty vs picking")
if rec_told is not None:
    rec_told.to_parquet(CLEAN / "cobot_reconciliation_told.parquet", index=False)

did_qty = "execqty" if "execqty" in result_tasks.columns else None
rec_did = reconcile_with_picking(result_tasks, picking, ident_result,
                                 qty_col=did_qty, slot_xw=slot_xw)
show(rec_did, "DID — OrderJobResultTask.execqty vs picking")
if rec_did is not None:
    rec_did.to_parquet(CLEAN / "cobot_reconciliation_did.parquet", index=False)

order_tasks.to_parquet(CLEAN / "cobot_order_tasks.parquet", index=False)
result_tasks.to_parquet(CLEAN / "cobot_result_tasks.parquet", index=False)
print("\ntask tables written to data/clean/")

negative-quantity instruction tasks: 107 (0.01%)
  by eventaction:
eventaction
NEW    107

=== TOLD — OrderJobTask.taskqty vs picking ===
  duplicate_task_emissions_collapsed: 20,768
  non_pick_tasks_excluded: 104,289
  qty_field_used: taskqty
  grain: (ticket, slot)
  (ticket, slot) pairs reconciled : 545,522
  robot units == WMS ordered units : 99.9%
  robot units == WMS picked units  : 99.1%

=== DID — OrderJobResultTask.execqty vs picking ===
  duplicate_task_emissions_collapsed: 571,832
  non_pick_tasks_excluded: 42,390
  qty_field_used: execqty
  grain: (ticket, slot)
  (ticket, slot) pairs reconciled : 545,067
  robot units == WMS ordered units : 98.9%
  robot units == WMS picked units  : 99.3%

task tables written to data/clean/


## 4. Duplicate task emissions

Counts repeat emissions per `(pickticket_id, jobtaskid)`. A rising collapse
count means the dedupe key is doing more work than it should.

In [5]:
# How often does the robot emit the same task more than once? Count rows per
# (ticket, task), then show the distribution and the eventtype mix.
rt = result_tasks.copy()
before = len(rt)
key = ["pickticket_id", "jobtaskid"]   # need both — jobtaskid alone repeats across tickets
dup_counts = rt.groupby(key, observed=True).size()

print(f"result_tasks rows      : {before:,}")
print(f"distinct (ticket,task) : {len(dup_counts):,}")
print(f"collapsed              : {before - len(dup_counts):,}")
print(f"\nemissions per task — distribution:")
print(dup_counts.value_counts().sort_index().head(10).to_string())

if "eventtype" in rt.columns:
    print(f"\nby eventtype:")
    print(rt["eventtype"].value_counts().head(8).to_string())

result_tasks rows      : 1,186,131
distinct (ticket,task) : 614,299
collapsed              : 571,832

emissions per task — distribution:
1     42467
2    571832

by eventtype:
eventtype
PICKCOMPLETE      614222
PICK              571909
CANCELCOMPLETE         0
CANCELREJECT           0
ACCEPT                 0
CANCEL                 0
NEW                    0
DUPTOTE                0


## Appendix — PICK vs PICKCOMPLETE disagreement

Not required downstream: `did_mismatch` comes from `taskqty` vs `execqty`, not
from these repeats. Kept as the evidence behind the register entry on
within-robot disagreement, with the CK cross-check baseline corrected per
D92. Set the flag to re-run.

In [6]:
# Appendix check: for tasks emitted twice (PICK, then PICKCOMPLETE), does the
# reported quantity differ between the two? And are the tickets where it
# differs more likely to be CKs than tickets in general? Set
# RUN_DISAGREEMENT = False to skip it on a routine re-run.
RUN_DISAGREEMENT = True

if RUN_DISAGREEMENT:
    pairs = rt[rt.duplicated(["pickticket_id", "jobtaskid"], keep=False)]
    piv = (pairs.pivot_table(index=["pickticket_id", "jobtaskid"],
                             columns="eventtype", values="execqty",
                             aggfunc="first", observed=True))
    piv = piv.dropna(subset=["PICK", "PICKCOMPLETE"])

    print(f"tasks with both events: {len(piv):,}")
    diff = piv["PICKCOMPLETE"] != piv["PICK"]
    print(f"execqty differs       : {int(diff.sum()):,}  ({diff.mean():.2%})")
    if diff.any():
        print(piv[diff].head(10).to_string())

    ck = pd.read_parquet(CLEAN / "cks.parquet")
    pt = pd.read_parquet(CLEAN / "pick_tickets.parquet")

    rt = result_tasks
    pairs = rt[rt.duplicated(["pickticket_id", "jobtaskid"], keep=False)]
    piv = (pairs.pivot_table(index=["pickticket_id", "jobtaskid"],
                             columns="eventtype", values="execqty",
                             aggfunc="first", observed=True)
                 .dropna(subset=["PICK", "PICKCOMPLETE"]))
    diff = piv["PICKCOMPLETE"] != piv["PICK"]

    disagree = piv[diff].reset_index()
    ck_tickets = set(ck["pickticket_id"].dropna().astype(str).str.upper())
    d_tickets = disagree["pickticket_id"].astype(str).str.upper()

    # The baseline CK rate uses the distinct ticket count, not the pre-dedupe row
    # count — the old baseline was wrong for exactly that reason (D92).
    n_tickets = pt["pickticket_id"].astype(str).str.upper().nunique()

    print(f"disagreeing tasks      : {len(disagree):,}")
    print(f"on distinct tickets    : {d_tickets.nunique():,}")
    print(f"tickets also in CK     : {d_tickets.isin(ck_tickets).sum():,}")
    print(f"CK rate on these       : {d_tickets.isin(ck_tickets).mean():.1%}")
    print(f"baseline CK ticket rate: {len(ck_tickets)/n_tickets:.1%}  "
          f"(over {n_tickets:,} tickets)")

    # Is the disagreement coming from one picker, one slot or one exception code?
    # Top 5 of each.
    t10 = disagree["pickticket_id"].astype(str).str.upper().unique()
    print("tickets:", len(t10))

    sub = rt[rt["pickticket_id"].astype(str).str.upper().isin(t10)]
    print(f"\ntotal result tasks on these tickets: {len(sub):,}")
    print(f"disagreeing share: {len(disagree)/len(sub):.1%}")

    for col in ["execuser", "_slot", "exceptioncode"]:
        if col in sub.columns:
            vc = sub[col].astype(str).value_counts()
            print(f"\n{col}: {vc.nunique()} distinct, top 5:")
            print(vc.head(5).to_string())

tasks with both events: 571,832
execqty differs       : 94  (0.02%)
eventtype                                                                            PICK  PICKCOMPLETE
pickticket_id                        jobtaskid                                                         
10AADF6C-9437-4967-84FA-5ACD8B6081B9 10aadf6c-9437-4967-84fa-5acd8b6081b9_156725_3      0             1
                                     10aadf6c-9437-4967-84fa-5acd8b6081b9_284445_4      0             1
                                     10aadf6c-9437-4967-84fa-5acd8b6081b9_492166_2      0             1
29A627C9-F61F-4D09-BD0C-6F796D2DB780 29a627c9-f61f-4d09-bd0c-6f796d2db780_468654_10     0             1
2C572964-D836-4322-B1C2-99CEBADF017E 2c572964-d836-4322-b1c2-99cebadf017e_254511_16     0             1
                                     2c572964-d836-4322-b1c2-99cebadf017e_297750_18     0             1
                                     2c572964-d836-4322-b1c2-99cebadf017e_327983_24     0           